# Average Audio Files

Load each experiment's `sync.csv`, then average microphone pairs for a range of experiments.

In [1]:
import ast
import platform
import re
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import wavfile

if platform.system() == "Windows":
    base_raw = Path(r"\\sanesstorage.cns.nyu.edu\archive\ginosar\Raw_data")
    base_processed = Path(r"\\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data")
else:
    base_raw = Path("/mnt/home/neurostatslab/ceph/saneslab_data/big_setup/")
    base_processed = Path("/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data")

print("Raw data base:", base_raw)
print("Processed data base:", base_processed)


Raw data base: /mnt/home/neurostatslab/ceph/saneslab_data/big_setup
Processed data base: /mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data


In [ ]:
def get_experiment_month(exp):
    if 97 <= exp <= 116:
        return "2024_12"
    elif 235 <= exp <= 237:
        return "2025_03"
    elif 272 <= exp <= 278:
        return "2025_07"
    elif 332 <= exp < 344:
        return "2025_10"
    elif exp >= 491:
        return "2026_02"
    else:
        raise ValueError(f"Unknown experiment range for {exp}")


def get_channel_mapping(exp):
    if exp < 272:
        return {
            "10": [0, 1],
            "20": [2, 3],
            "30": [4, 5],
        }
    return {
        "10": [2, 3],
        "20": [4, 5],
        "30": [0, 1],
    }


def build_experiment_paths(exp):
    month_folder = get_experiment_month(exp)
    experiment_root = base_raw / f"experiment_{exp}"
    folder_path_raw_wavs = experiment_root / "concatenated_data_cam_mic_sync"
    folder_path_sync = folder_path_raw_wavs
    folder_path_averaged_wavs = base_processed / "Audio" / month_folder / str(exp) / "Averaged_wavs_w_annotations"
    folder_path_audio = base_processed / "Audio" / month_folder / str(exp)
    return {
        "month_folder": month_folder,
        "experiment_root": experiment_root,
        "raw_wavs": folder_path_raw_wavs,
        "sync": folder_path_sync,
        "averaged_wavs": folder_path_averaged_wavs,
        "audio": folder_path_audio,
    }


def load_sync_file(exp, folder_path_sync):
    sync_path = Path(folder_path_sync) / "sync.csv"
    if not sync_path.exists():
        raise FileNotFoundError(f"Sync file not found: {sync_path}")

    sync_df = pd.read_csv(sync_path)
    print(list(sync_df.columns))

    sync_df["timestamp"] = sync_df["timestamp"].apply(ast.literal_eval)
    sync_df["start_time"] = pd.to_datetime(sync_df["timestamp"].str[0])
    start_time_experiment = sync_df.iloc[0]["start_time"]
    print(f"Experiment {exp} started at: {start_time_experiment}")

    return start_time_experiment, sync_df


def average_microphone_pairs(exp, input_folder, output_folder):
    input_folder = Path(input_folder)
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    channel_mapping = get_channel_mapping(exp)

    print(f"\n--- Processing experiment {exp} ---")
    print(f"Reading raw WAV files from: {input_folder}")
    print(f"Saving averaged WAV files to: {output_folder}")

    for virtual_ch, real_pair in channel_mapping.items():
        print(f"  virtual channel {virtual_ch} <- real channels {real_pair}")

    file_nums = []
    pattern = re.compile(r"channel_00_file_(\d+)\.wav")

    for file_path in input_folder.iterdir():
        match = pattern.match(file_path.name)
        if match:
            file_nums.append(int(match.group(1)))

    file_nums = sorted(file_nums)
    print(f"Found {len(file_nums)} file chunks")

    n_saved = 0
    n_missing = 0
    n_shape_mismatch = 0
    n_rate_mismatch = 0

    for file_num in file_nums:
        file_num_str = f"{file_num:03d}"

        for virtual_ch, real_pair in channel_mapping.items():
            ch1, ch2 = real_pair
            path1 = input_folder / f"channel_{ch1:02d}_file_{file_num_str}.wav"
            path2 = input_folder / f"channel_{ch2:02d}_file_{file_num_str}.wav"

            if not path1.exists() or not path2.exists():
                print("  Skipping because one or both files are missing")
                print(f"  Expected file: {path1.name}")
                print(f"  Expected file: {path2.name}")
                n_missing += 1
                continue

            rate1, data1 = wavfile.read(path1)
            rate2, data2 = wavfile.read(path2)

            if rate1 != rate2:
                print(f"  Skipping because sample rates differ: {rate1} vs {rate2}")
                n_rate_mismatch += 1
                continue

            if data1.shape != data2.shape:
                print(f"  Skipping because shapes differ: {data1.shape} vs {data2.shape}")
                n_shape_mismatch += 1
                continue

            avg_data = (data1.astype(np.float32, copy=False) + data2.astype(np.float32, copy=False)) / 2.0
            out_path = output_folder / f"channel_{virtual_ch}_file_{file_num_str}.wav"
            wavfile.write(out_path, rate1, avg_data)
            n_saved += 1

    print("\n--- Done ---")
    print(f"Saved files: {n_saved}")
    print(f"Skipped because of missing files: {n_missing}")
    print(f"Skipped because of shape mismatch: {n_shape_mismatch}")
    print(f"Skipped because of sample rate mismatch: {n_rate_mismatch}")

    return {
        "saved_files": n_saved,
        "missing_pairs": n_missing,
        "shape_mismatch": n_shape_mismatch,
        "rate_mismatch": n_rate_mismatch,
    }


def copy_experiment_log_file(exp, experiment_root, destination_folder):
    experiment_root = Path(experiment_root)
    destination_folder = Path(destination_folder)
    destination_folder.mkdir(parents=True, exist_ok=True)

    log_files = sorted(experiment_root.glob(f"experiment_{exp}_log*.txt"))
    if not log_files:
        raise FileNotFoundError(f"No log txt file found in {experiment_root}")

    log_path = log_files[0]
    destination_path = destination_folder / log_path.name
    shutil.copy2(log_path, destination_path)
    print(f"Copied log file to: {destination_path}")
    return destination_path


In [ ]:
# Set the experiment range here.
start_exp = 514
end_exp = 538
experiment_ids = list(range(start_exp, end_exp + 1))

# Optional: use a manual list instead.
# experiment_ids = [491, 492, 518]

stop_on_error = False
results = []

for exp in experiment_ids:
    print(f"\n{'=' * 80}")
    print(f"Starting experiment {exp}")

    try:
        paths = build_experiment_paths(exp)
        print("Raw WAV folder:", paths["raw_wavs"])
        print("Sync folder:", paths["sync"])
        print("Output folder:", paths["averaged_wavs"])
        print("Processed experiment folder:", paths["audio"])

        start_time_experiment, sync_df = load_sync_file(exp, paths["sync"])
        log_copy_path = copy_experiment_log_file(exp, paths["experiment_root"], paths["audio"])
        summary = average_microphone_pairs(exp, paths["raw_wavs"], paths["averaged_wavs"])

        results.append({
            "exp": exp,
            "month_folder": paths["month_folder"],
            "start_time_experiment": start_time_experiment,
            "sync_rows": len(sync_df),
            "copied_log_file": str(log_copy_path),
            **summary,
        })
    except Exception as exc:
        print(f"Failed for experiment {exp}: {exc}")
        results.append({"exp": exp, "error": str(exc)})
        if stop_on_error:
            raise

results_df = pd.DataFrame(results)
results_df



Starting experiment 336
Raw WAV folder: /mnt/home/neurostatslab/ceph/saneslab_data/big_setup/experiment_336/concatenated_data_cam_mic_sync
Sync folder: /mnt/home/neurostatslab/ceph/saneslab_data/big_setup/experiment_336/concatenated_data_cam_mic_sync
Output folder: /mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio/2025_10/336/Averaged_wavs_w_annotations
Processed experiment folder: /mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio/2025_10/336
['video', 'audio', 'timestamp']
Experiment 336 started at: 2025-10-09 15:09:25
Copied log file to: /mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio/2025_10/336/experiment_336_log_2025-10-09_15-09-09.txt

--- Processing experiment 336 ---
Reading raw WAV files from: /mnt/home/neurostatslab/ceph/saneslab_data/big_setup/experiment_336/concatenated_data_cam_mic_sync
Saving averaged WAV files to: /mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio/2025_10/33